# 01 — NTU60 2D skeleton · ST-GCN baseline (smoke test)

Pipeline milestone:

```
NTU60 2D skeleton  ->  MMAction2 PoseDataset  ->  ST-GCN  ->  60-class prediction
                   ->  loss + val metric      ->  checkpoint
```

**Scope:** Tasks 1–12 of the baseline plan ONLY (xsub split, 1-epoch smoke run,
checkpoint + validation). No YOLO/MMPose/tracking/TensorRT/deployment, no full
80-epoch training here.

**Kaggle settings required:** Accelerator = GPU (T4/P100), Internet = ON.

Run cells top-to-bottom. Everything lands under `/kaggle/working/ntu-action-recognition`.


In [ ]:
# Setup — project layout (clones your repo, or uses files you uploaded)
from pathlib import Path

REPO_URL     = 'https://github.com/mzuyyy/Human-action-recognition.git'  # change to your fork if needed
PROJECT_DIR  = Path('/kaggle/working/ntu-action-recognition')
MMACTION2_DIR= Path('/kaggle/working/mmaction2')
WORK_DIR     = PROJECT_DIR / 'work_dirs/stgcn_smoke_test'
NTU60_URL    = 'https://download.openmmlab.com/mmaction/v1.0/skeleton/data/ntu60_2d.pkl'

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
for d in ['data/skeleton', 'artifacts/skeleton_samples', 'configs',
          'scripts', 'notebooks', str(WORK_DIR)]:
    Path(d).mkdir(parents=True, exist_ok=True)
print('project ready at', PROJECT_DIR)


In [ ]:
# Task 1 — environment check (saved to artifacts/environment.txt)
import shutil, subprocess, sys
import torch

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip() or '(no output)'

lines = ['$ nvidia-smi']
lines.append(sh('nvidia-smi') if shutil.which('nvidia-smi') else 'nvidia-smi not found')
lines += [
    '',
    'Python: ' + sys.version,
    'PyTorch: ' + torch.__version__,
    'CUDA runtime: ' + str(torch.version.cuda),
    'CUDA available: ' + str(torch.cuda.is_available()),
]
if torch.cuda.is_available():
    lines.append('GPU: ' + torch.cuda.get_device_name(0))
    lines.append('GPU count: ' + str(torch.cuda.device_count()))

env_txt = '\n'.join(lines)
print(env_txt)
env_file = PROJECT_DIR / 'artifacts/environment.txt'
env_file.write_text(env_txt + '\n')
print('\nsaved ->', env_file)
# NOTE: do NOT upgrade/downgrade PyTorch here — OpenMMLab stack is installed against this one.


In [ ]:
%%bash
# Task 2 - OpenMMLab stack. MMDetection/MMPose intentionally NOT installed.
# Always drive installs through `python -m pip`: bare `pip` may belong to a
# DIFFERENT interpreter (conda vs /usr/local) than the notebook kernel.
set -e
python -m pip install --force-reinstall -q -U pip setuptools wheel
echo "setuptools -> $(python -c 'import setuptools; print(setuptools.__version__, setuptools.__file__)')"

python -m pip install -q mmengine

# prebuilt mmcv wheel matching the PREINSTALLED kaggle torch/cuda;
# if no wheel exists for this combo, pip falls back to a ~20-35 min source build.
TORCH_TAG=$(python -c "import torch; print('torch' + '.'.join(torch.__version__.split('+')[0].split('.')[:2]))")
CUDA_TAG=$(python -c "import torch; v=torch.version.cuda; print('cu' + ''.join(v.split('.')[:2]))")
echo "prebuilt mmcv index for ${CUDA_TAG} / ${TORCH_TAG}"
python -m pip install "mmcv>=2.0.0" -f "https://download.openmmlab.com/mmcv/dist/${CUDA_TAG}/${TORCH_TAG}/index.html"


In [ ]:
%%bash
# Task 2b - mmaction2 itself (tools/train.py + configs live in the repo).
set -e
rm -rf /kaggle/working/mmaction2
git clone --depth 1 https://github.com/open-mmlab/mmaction2.git /kaggle/working/mmaction2
python -m pip install -q -e /kaggle/working/mmaction2


In [ ]:
# Task 2 — acceptance criterion: all four imports succeed
import mmaction, mmcv, mmengine, torch

print('torch   :', torch.__version__)
print('mmengine:', mmengine.__version__)
print('mmcv    :', mmcv.__version__)
print('mmaction:', mmaction.__version__)
assert torch.cuda.is_available(), 'GPU accelerator is OFF (Settings -> Accelerator)'
with open(PROJECT_DIR / 'artifacts/environment.txt', 'a') as f:
    f.write(f'MMEngine: {mmengine.__version__}\n'
            f'MMCV: {mmcv.__version__}\n'
            f'MMAction2: {mmaction.__version__}\n')


In [ ]:
# Task 3 — fetch ntu60_2d.pkl (official OpenMMLab release, ~1.4 GB).
# Internet OFF fallback: symlink an attached /kaggle/input copy instead.
import glob, os
import urllib.request

dst = PROJECT_DIR / 'data/skeleton/ntu60_2d.pkl'
if dst.exists():
    print('already present:', dst)
else:
    hits = glob.glob('/kaggle/input/**/ntu60_2d.pkl', recursive=True)
    if hits:
        os.symlink(os.path.abspath(hits[0]), dst)
        print('symlinked attached dataset:', hits[0])
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(NTU60_URL, dst)
        print(f'downloaded -> {dst} ({dst.stat().st_size/1e9:.2f} GB)')


In [ ]:
%%bash
# Task 4 — inspect the pickle BEFORE training
python scripts/inspect_ntu60.py \
    --ann-file data/skeleton/ntu60_2d.pkl \
    --json-out artifacts/dataset_stats.json


In [ ]:
# Task 5 — visualize skeletons (3 samples x 5 frames, COCO-17 on white canvas)
!python scripts/visualize_skeleton.py --ann-file data/skeleton/ntu60_2d.pkl --out-dir artifacts/skeleton_samples

from IPython.display import Image, display
display(Image(filename=str(PROJECT_DIR / 'artifacts/skeleton_samples/skeleton_samples_overview.png')))
# If skeletons look wrong (detached joints, mirrored bodies, static blobs across frames)
# STOP HERE — do not train on broken preprocessing.


## Tasks 6–7 — configs

* `configs/stgcn_ntu60_xsub_baseline.py` — flattened copy of the official
  `stgcn_8xb16-joint-u100-80e_ntu60-xsub-keypoint-2d.py` (`_base_/default_runtime.py`
  inlined). 80 epochs, joint representation, COCO layout, clip_len=100, RepeatDataset×5.
* `configs/stgcn_ntu60_xsub_smoke.py` — inherits the baseline, overrides for a
  single-GPU smoke run: `max_epochs=1`, `times=1`, `batch_size=16`, `num_workers=2`,
  `lr=0.01`, checkpoint on. Seed 42 is passed on the CLI.


In [ ]:
# Task 6/7 sanity — resolve the smoke config and show effective values
from mmengine.config import Config

cfg = Config.fromfile(str(PROJECT_DIR / 'configs/stgcn_ntu60_xsub_smoke.py'))
tr = cfg.train_dataloader.dataset
pipe = tr.dataset.pipeline
print('model          :', cfg.model.type, '/', cfg.model.backbone.type)
print('ann_file       :', tr.dataset.ann_file)
print('splits         : train =', tr.dataset.split, '| val =', cfg.val_dataloader.dataset.split)
print('representation :', [p['feats'] for p in pipe if p['type'] == 'GenSkeFeat'][0])
print('layout         :', cfg.model.backbone.graph_cfg.layout, '(17 joints)')
print('clip_len       :', [p['clip_len'] for p in pipe if p['type'] == 'UniformSampleFrames'][0])
print('RepeatDataset  : times =', tr.times)
print('batch/workers  :', cfg.train_dataloader.batch_size, '/', cfg.train_dataloader.num_workers)
print('epochs / lr    :', cfg.train_cfg.max_epochs, '/', cfg.optim_wrapper.optimizer.lr)
print('num_classes    :', cfg.model.cls_head.num_classes)


In [ ]:
# Task 8 — load ONE batch through config-built dataset + model preprocessing
import torch
from mmengine.config import Config
from mmengine.registry import init_default_scope
from mmengine.dataset import pseudo_collate
from mmaction.registry import DATASETS, MODELS
from torch.utils.data import DataLoader

init_default_scope('mmaction')
cfg = Config.fromfile(str(PROJECT_DIR / 'configs/stgcn_ntu60_xsub_smoke.py'))

dataset = DATASETS.build(cfg.train_dataloader.dataset)
loader = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0,   # workers=0 in-notebook on purpose
                    collate_fn=pseudo_collate)
model = MODELS.build(cfg.model).to('cuda' if torch.cuda.is_available() else 'cpu')

batch = next(iter(loader))
proc = model.data_preprocessor(batch, training=True)
inputs, data_samples = proc['inputs'], proc['data_samples']
labels = torch.tensor([int(s.gt_labels.label) for s in data_samples])

print('input  shape:', tuple(inputs.shape), '| dtype:', inputs.dtype, '| device:', inputs.device)
print('labels shape:', tuple(labels.shape), '| dtype:', labels.dtype, '| values:', labels.tolist())

assert inputs.numel() > 0,                     'empty input tensor'
assert torch.isfinite(inputs).all(),           'NaN/Inf found in inputs'
assert labels.min() >= 0 and labels.max() < 60, 'label out of range'
print('PASS — 1 batch loaded cleanly')


In [ ]:
# Task 9 — smoke training: 1 epoch on GPU, monitored (utilization + VRAM)
import subprocess, sys, threading, time

gpu_log, stop = [], threading.Event()

def watch():
    while not stop.is_set():
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True).stdout.strip()
        if out:
            gpu_log.append(out.splitlines()[0])
        time.sleep(5)

t = threading.Thread(target=watch, daemon=True)
t.start()
t0 = time.time()
p = subprocess.run([sys.executable, 'tools/train.py',
                    'configs/stgcn_ntu60_xsub_smoke.py',
                    '--work-dir', 'work_dirs/stgcn_smoke_test',
                    '--seed', '42'],
                   cwd=str(PROJECT_DIR), capture_output=True, text=True)
TRAIN_SECONDS = time.time() - t0
stop.set(); t.join()

print('\n'.join(p.stdout.splitlines()[-45:]))
if p.returncode != 0:
    print(p.stderr[-4000:])
    raise RuntimeError(f'training failed, rc={p.returncode}')

(PROJECT_DIR / 'artifacts/gpu_monitor.csv').write_text(
    'util_pct,mem_mb\n' + '\n'.join(gpu_log) + '\n')
utils = [int(x.split(',')[0]) for x in gpu_log if ',' in x]
mems  = [int(x.split(',')[1]) for x in gpu_log if ',' in x]
print(f'\ntrain wall time: {TRAIN_SECONDS:.0f}s | '
      f'avg GPU util: {sum(utils)/max(len(utils),1):.0f}% | '
      f'peak VRAM: {max(mems) if mems else 0} MB')
ckpts = sorted(x.name for x in WORK_DIR.glob('*.pth'))
print('checkpoints:', ckpts)
assert any('epoch_1' in c or 'latest' in c for c in ckpts), 'no checkpoint written'
# OOM? edit configs/stgcn_ntu60_xsub_smoke.py: batch_size 16 -> 8 -> 4 (never shrink the model)


In [ ]:
# Task 10 — validation results (AccMetric logged during training on xsub_val)
import json

val_acc, last_loss, last_lr = {}, None, None
for logf in sorted(WORK_DIR.rglob('*.json')):
    if 'scalars' not in logf.name:
        continue
    for line in logf.read_text().splitlines():
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if rec.get('mode') == 'train' and 'loss' in rec:
            last_loss, last_lr = rec['loss'], rec.get('lr')
        elif rec.get('mode') == 'val':
            val_acc.update({k: round(v, 4) for k, v in rec.items()
                            if isinstance(v, (int, float)) and k.startswith('acc')})
print('final train loss:', last_loss, '| lr:', last_lr)
print('validation:', val_acc)
assert 'acc_top1' in val_acc, 'no Top-1 metric found in logs'

# Optional: standalone eval on the checkpoint (10 clips/sample — much slower):
# !python tools/test.py configs/stgcn_ntu60_xsub_smoke.py work_dirs/stgcn_smoke_test/epoch_1.pth --work-dir work_dirs/stgcn_smoke_test


In [ ]:
# Task 12 — build the post-smoke-test report -> artifacts/smoke_report.txt
import json, platform

stats = json.load(open(PROJECT_DIR / 'artifacts/dataset_stats.json'))
env_lines = (PROJECT_DIR / 'artifacts/environment.txt').read_text().splitlines()
env = {}
for ln in env_lines:
    if ':' in ln and not ln.startswith('$'):
        k, _, v = ln.partition(':')
        env[k.strip()] = v.strip()

report = f"""ENVIRONMENT
-----------
GPU:        {env.get('GPU', 'n/a')}
Python:     {platform.python_version()}
PyTorch:    {env.get('PyTorch', '')}
CUDA:       {env.get('CUDA runtime', '')}
MMCV:       {env.get('MMCV', '')}
MMEngine:   {env.get('MMEngine', '')}
MMAction2:  {env.get('MMAction2', '')}

DATASET
-------
File:                  data/skeleton/ntu60_2d.pkl
Number of samples:     {stats['num_annotations']}
Number of classes:     {stats['num_classes']}
xsub_train:            {stats['splits'].get('xsub_train')}
xsub_val:              {stats['splits'].get('xsub_val')}
Example keypoint shape:       {tuple(stats['example_keypoint_shape'])}  (M x T x V x C)
Example keypoint_score shape: {tuple(stats['example_keypoint_score_shape'])}  (M x T x V)

MODEL
-----
Model:                ST-GCN (RecognizerGCN + STGCN backbone, COCO layout)
Input representation: joint
Number of classes:    {stats['num_classes']}
Batch size:           {cfg.train_dataloader.batch_size}

SMOKE TRAIN
-----------
Epochs:          {cfg.train_cfg.max_epochs}
Train time:      {TRAIN_SECONDS:.0f}s
Final train loss:{last_loss}
Peak GPU memory: {max(mems) if mems else 0} MB

VALIDATION
----------
Top-1: {val_acc.get('acc_top1')}
Top-5: {val_acc.get('acc_top5')}

OUTPUTS
-------
Checkpoint:             {WORK_DIR}/epoch_1.pth (+ best/latest)
Config:                 configs/stgcn_ntu60_xsub_baseline.py + _smoke.py
Logs:                   {WORK_DIR}/<timestamp>/
Skeleton visualization: artifacts/skeleton_samples/

ISSUES
------
1. (fill in if any)
2.

NEXT RECOMMENDED STEP
---------------------
Smoke pipeline verified end-to-end -> launch the full 80-epoch baseline
(configs/stgcn_ntu60_xsub_baseline.py) on a fresh Kaggle session.
"""
(PROJECT_DIR / 'artifacts/smoke_report.txt').write_text(report)
print(report)


## Stop here — Definition of Done check

Milestone complete when ALL six hold (verify above):

1. ✅ `import mmaction` succeeds on Kaggle GPU
2. ✅ `ntu60_2d.pkl` loads
3. ✅ dataset stats + tensor shapes printed (Task 4)
4. ✅ skeleton visualization looks sane (Task 5)
5. ✅ ST-GCN smoke run finished on GPU, finite loss, checkpoint written
6. ✅ Top-1 / Top-5 recorded on `xsub_val`

**Do not start the full 80-epoch run until you've reviewed the report.**
When ready: new session -> run the same notebook but replace the training cell with
`python tools/train.py configs/stgcn_ntu60_xsub_baseline.py --work-dir work_dirs/stgcn_full --seed 42`
(budget ~9-12 h on a P100; consider Kaggle GPU-quota limits).
